# Visualización de Persistencia Homológica

Steffany Mishell Lara Muy        |  A00838589 

Yamil Elías del Blanco Chávez    |  A00838610

Paola Michelle Martínez Galeazzi |  A00839082


## A) Visión Industrial: Inspección de Defectos

### El juguete de Intuición

**Instrucciones**:


El Juguete de Intuición (15 min)
Antes de cargar imágenes reales, programen la intuición:

1. Creen dos matrices (imágenes) en numpy de 10x10.
2. Imagen A: Un cuadrado sólido (lleno de 1s).
3. Imagen B: Un cuadrado sólido con un bloque central de 3x3 lleno de 0s (un agujero).
4. Apliquen CubicalPersistence y grafiquen el Barcode.

* Observación: La Imagen B mostrará una barra larga en H₁ (agujero). ¡Detectaron un defecto sin redes neuronales!

In [11]:
# ------------ Importar Librerías
import numpy as np
#import pandas as pd
#import gudhi #este no era, me equivoqué
import gtda
from gtda.homology import CubicalPersistence



In [12]:
import sklearn
import gtda
import numpy as np

print("sklearn:", sklearn.__version__)
print("gtda:", gtda.__version__)
print("numpy:", np.__version__)

sklearn: 1.3.2
gtda: 0.6.2
numpy: 1.26.4


In [13]:
# Paso 1-2
imagenA=np.ones((10,10))
imagenB=np.ones((10,10))

# Paso 2, con centro (3x3) de 0s
## 10x10, centro 
imagenB[4:7,4:7]=0
imagenB

array([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 0., 0., 0., 1., 1., 1.],
       [1., 1., 1., 1., 0., 0., 0., 1., 1., 1.],
       [1., 1., 1., 1., 0., 0., 0., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [14]:
# cambiamos la imagen shape, para decirle que solo es 1

#imagenA.reshape(1,10,10)
#imagenB.reshape(1,10,10)

#procesaré juntas las imagenes porque es la misma funcion sorta

#apilar imagenes
stackImagenes = np.stack([imagenA,imagenB]).astype(np.float32)
#.astype(np.float32) porque mis imagenes eran int cuando las creé con la función
# de numpy

#crear obj
cubPersistence = CubicalPersistence(homology_dimensions = [0,1])
diagramas = cubPersistence.fit_transform(stackImagenes)

'''

- acá me equivoqué porque habia usado gudhi en lugar de gtda
cubPersistenceA = CubicalPersistence(homology_dimensions=)

cubPersistenceA.compute_persistence(persistence_dim_max=True)

cubPersistenceB = gudhi.CubicalComplex(
    dimensions = [10,10],
    top_dimensional_cells = imagenB.flatten()
)
cubPersistenceB.compute_persistence()
'''

'\n\n- acá me equivoqué porque habia usado gudhi en lugar de gtda\ncubPersistenceA = CubicalPersistence(homology_dimensions=)\n\ncubPersistenceA.compute_persistence(persistence_dim_max=True)\n\ncubPersistenceB = gudhi.CubicalComplex(\n    dimensions = [10,10],\n    top_dimensional_cells = imagenB.flatten()\n)\ncubPersistenceB.compute_persistence()\n'

In [18]:
print(diagramas)

[[[0. 0. 0.]
  [0. 0. 1.]]

 [[0. 0. 0.]
  [0. 0. 1.]]]


### El Reto Principal

**Instrucciones**:

Procesamiento del Dataset de Kaggle:

- Tomen una muestra de 300 imágenes sanas y 300 defectuosas.
- Vital: Reduzcan la resolución a 32x32 píxeles (para que el complejo cúbico corra en segundos).
- Instancien CubicalPersistence y transformen los diagramas resultantes usando PersistenceImage.
- Aplánenlos para obtener un vector 1D por cada imagen.

### Benchmark: TDA vs Clásico

Entrenen un RandomForestClassifier comparando los enfoques:

1. **Modelo A (Clásico):**
Entrenado usando únicamente los píxeles aplanados (1024 variables).

2. **Modelo B (Topológico):**
Entrenado usando únicamente las Imágenes de Persistencia (Vectores Topológicos).

### El "Aha Moment" (Test de Robustez)

Probablemente, ambos modelos obtengan un Accuracy similar (ej. 85%). Para demostrar el poder del TDA, ejecuten este test:

- **Reto**: Roten todas las imágenes del set de prueba (X_test) 90 grados, o gírenlas verticalmente (Flip). Vuelvan a evaluar ambos modelos.
- **Resultado**: El Modelo Clásico colapsará drásticamente (los píxeles se movieron). El Modelo TDA mantendrá su precisión casi intacta, porque un agujero sigue siendo un agujero, sin importar cómo rotes la pieza.